# धडा 11 - मॉडेल संदर्भ प्रोटोकॉल (MCP)

**मॉडेल संदर्भ प्रोटोकॉल (MCP)** हा एक खुला मानक आहे जो एजंटांना रनटाइमवर डायनॅमिक पद्धतीने साधने, संसाधने आणि डेटा स्त्रोत शोधून ते वापरण्याची परवानगी देतो. एजंटमध्ये साधने हार्डकोड करण्याऐवजी, MCP एजंटांना बाह्य सर्व्हरशी कनेक्ट होण्याची परवानगी देते जे आवश्यकतेनुसार क्षमता उघडतात.

या धड्यात, तुम्ही शिकाल:
- MCP म्हणजे काय आणि एजंट प्रणालींसाठी ते का महत्त्वाचे आहे
- MCP क्लायंट-सर्व्हर आर्किटेक्चर कसे कार्य करते
- MCP-शैलीच्या साधन शोधाचा वापर करणारे एजंट कसे तयार करायचे


## सेटअप

**पूर्वअटी:**
- Azure AI Foundry प्रकल्प ज्यात तैनात केलेले मॉडेल आहे
- प्रमाणीकरणासाठी `az login` चालवा


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## मॉडेल संदर्भ प्रोटोकॉल (MCP) काय आहे?

MCP AI एजंट्सना बाह्य साधने आणि डेटा स्रोत शोधण्यासाठी आणि त्यांच्याशी संवाद साधण्यासाठी एक मानक पद्धत परिभाषित करते:

- **MCP सर्व्हर**: प्रमाणित प्रोटोकॉलद्वारे साधने, संसाधने आणि प्रॉम्प्ट उपलब्ध करून देते
- **MCP क्लायंट**: एजंट रनटाइम जे सर्व्हरशी कनेक्ट होते आणि उपलब्ध क्षमता शोधून काढते
- **गतिशील शोध**: एजंट्सना हार्डकोड केलेल्या साधनांची गरज नसते — ते रनटाइमवर काय उपलब्ध आहे ते शोधून काढतात

हे विस्तारक्षम एजंट सिस्टम्स तयार करण्यासाठी शक्तिशाली आहे जिथे नवीन क्षमता एजंट कोडमध्ये बदल न करता जोडता येतात.


## MCP कसे कार्य करते

```
┌─────────────┐     discover      ┌─────────────────┐
│  MCP Client  │ ──────────────► │   MCP Server     │
│  (Agent)     │                  │  (Tool Provider) │
│              │ ◄────────────── │                   │
│              │   tool results   │  • list_tools()  │
│              │                  │  • call_tool()   │
└─────────────┘                  │  • resources     │
                                  └─────────────────┘
```

1. एजंट (MCP client) एका MCP सर्व्हरशी कनेक्ट होतो
2. सर्व्हर उपलब्ध साधने आणि त्यांच्या स्कीमांची यादी परत पाठवते
3. एजंट नंतर त्याच्या विचार प्रक्रिये दरम्यान शोधलेल्या कोणत्याही साधनाला कॉल करू शकतो
4. निकाल तेच प्रोटोकॉल वापरून परत येतात


## MCP टूल शोधाचे अनुकरण

वास्तविक MCP सर्व्हरसाठी चालू असलेली सर्व्हर प्रक्रिया आवश्यक असल्यामुळे, आम्ही `@tool` फंक्शन्स वापरून हा नमुना दाखवणार आहोत जे MCP-शी जोडलेल्या आवास सेवेने काय प्रदान केले असते ते अनुकरण करतात.

उत्पादनात, ही साधने स्थानिकरित्या परिभाषित करण्याऐवजी MCP सर्व्हरकडून गतिशीलरित्या शोधली जातील.


In [ ]:
@tool(approval_mode="never_require")
def search_accommodations(
    location: Annotated[str, "The city to search for accommodations"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"] = 2
) -> str:
    """Search for accommodations (simulating an MCP-connected Airbnb tool).
    In production, this would be discovered via MCP from an accommodation service."""
    listings = {
        "Tokyo": [
            {"name": "Shinjuku Modern Apartment", "price": 120, "rating": 4.8},
            {"name": "Traditional Ryokan in Asakusa", "price": 200, "rating": 4.9},
            {"name": "Shibuya Studio", "price": 85, "rating": 4.5},
        ],
        "Paris": [
            {"name": "Le Marais Charming Flat", "price": 150, "rating": 4.7},
            {"name": "Montmartre Artist Loft", "price": 110, "rating": 4.6},
        ],
        "Barcelona": [
            {"name": "Gothic Quarter Penthouse", "price": 130, "rating": 4.8},
            {"name": "Barceloneta Beach Flat", "price": 95, "rating": 4.4},
        ],
    }
    results = listings.get(location, [])
    if not results:
        return f"No accommodations found in {location}"
    output = f"Accommodations in {location} ({check_in} to {check_out}, {guests} guests):\n"
    for listing in results:
        output += f"  - {listing['name']}: ${listing['price']}/night (★{listing['rating']})\n"
    return output


@tool(approval_mode="never_require")
def get_local_experiences(
    location: Annotated[str, "The city to find experiences in"],
    interest: Annotated[str, "Type of experience (food, culture, adventure, etc.)"] = "all"
) -> str:
    """Get local experiences and activities (simulating an MCP-connected tourism tool)."""
    experiences = {
        "Tokyo": {
            "food": ["Tsukiji Market Tour ($45)", "Ramen Making Class ($60)", "Sake Tasting ($35)"],
            "culture": ["Tea Ceremony ($50)", "Samurai Museum ($15)", "Sumo Tournament ($80)"],
            "adventure": ["Mt. Fuji Day Trip ($120)", "Go-kart City Tour ($80)"],
        },
        "Paris": {
            "food": ["Wine & Cheese Tasting ($55)", "Cooking Class ($90)", "Market Tour ($40)"],
            "culture": ["Louvre Guided Tour ($35)", "Montmartre Art Walk ($25)"],
        },
    }
    city_exp = experiences.get(location, {})
    if not city_exp:
        return f"No experiences found in {location}"
    if interest != "all" and interest in city_exp:
        items = city_exp[interest]
        return f"{interest.title()} experiences in {location}:\n" + "\n".join(f"  - {e}" for e in items)
    output = f"All experiences in {location}:\n"
    for cat, items in city_exp.items():
        output += f"\n  {cat.title()}:\n"
        for item in items:
            output += f"    - {item}\n"
    return output

## MCP-शैलीच्या साधनांसह एजंट तयार करणे


In [ ]:
agent = await provider.create_agent(
    tools=[search_accommodations, get_local_experiences],
    name="AccommodationAgent",
    instructions="""You are an accommodation and travel experiences specialist powered by MCP-connected services.

Help travelers find the perfect place to stay and things to do. When searching:
1. Use the search_accommodations tool to find listings
2. Use the get_local_experiences tool to suggest activities
3. Compare options and make personalized recommendations
4. Consider the traveler's budget, interests, and travel style""",
)

response = await agent.run(
    "I'm visiting Tokyo for 5 nights in April with my partner. We love traditional Japanese culture and food. "
    "Find us a place to stay and suggest some experiences.",
    )
print(response)

## MCP उत्पादनात

उत्पादन वातावरणात, MCP प्रभावी पद्धती सक्षम करते:

- **Dynamic tool discovery**: एजंट्स MCP सर्व्हरशी कनेक्ट होतात आणि रनटाइममध्ये साधने शोधतात
- **Decoupled architecture**: टूल पुरवठादार एजंटपासून स्वतंत्रपणे अद्यतन करू शकतात
- **Cross-organization sharing**: संघ MCP सर्व्हरद्वारे क्षमतांची माहिती उघड करू शकतात जी कोणताही एजंट वापरू शकतो
- **Microsoft Agent Framework support**: MAF मध्ये `mcp` इंटिग्रेशनद्वारे अंगभूत MCP क्लायंट समर्थन आहे

MAF सोबत प्रत्यक्ष MCP सर्व्हर वापरण्यासाठी, आपण `hosted_mcp_tool()` किंवा MCP क्लायंट इंटिग्रेशनद्वारे कनेक्ट करता.

**अधिक जाणून घ्या:**
- [MCP Specification](https://modelcontextprotocol.io/)
- [Microsoft Agent Framework MCP Support](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/mcp)


## सारांश

- **MCP** एजंट्स आणि साधन पुरवठादारांदरम्यान डायनॅमिक साधने शोधण्यासाठी एक मुक्त मानक आहे
- **क्लायंट-सर्व्हर आर्किटेक्चर** एजंट्सना रनटाइममध्ये क्षमता शोधण्याची परवानगी देते
- MCP सक्षम करते **विस्तारयोग्य, विभक्त एजंट प्रणाली** जिथे साधने कोड बदलल्याशिवाय जोडली जाऊ शकतात
- Microsoft Agent Framework उत्पादन वापरासाठी **बिल्ट-इन MCP समर्थन** प्रदान करते


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
अस्वीकरण:
हे दस्तऐवज AI अनुवाद सेवा Co-op Translator (https://github.com/Azure/co-op-translator) वापरून अनुवादित केले आहे. आम्ही अचूकतेसाठी प्रयत्न करतो, परंतु कृपया लक्षात ठेवा की स्वयंचलित अनुवादांमध्ये त्रुटी किंवा अचूकतेशी संबंधित चुका असू शकतात. मूळ दस्तऐवज त्याच्या मूळ भाषेत अधिकृत स्रोत म्हणून विचारात घ्यावा. महत्त्वाच्या माहितीच्या बाबतीत व्यावसायिक मानवी अनुवाद करणे सुचविले जाते. या अनुवादाच्या वापरामुळे उद्भवणाऱ्या कोणत्याही गैरसमजुती किंवा चुकीच्या अर्थनिर्वचनांसाठी आम्ही जबाबदार नाही.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
